
# 28 — Satellite Opportunity Scanner

This notebook scans for the best Sentinel-5P observation opportunities for Newhaven ERF by combining:
- Sentinel-5P catalogue coverage
- historical cloud cover
- historical wind speed
- a simple screening score

Outputs:
- outputs/opportunity_scanner/opportunity_scene_catalog.csv
- outputs/opportunity_scanner/opportunity_weather_hourly.csv
- outputs/opportunity_scanner/opportunity_scene_evaluation.csv
- outputs/opportunity_scanner/opportunity_scan_daily.csv
- outputs/opportunity_scanner/opportunity_scan_ranked.csv
- outputs/opportunity_scanner/opportunity_scan_plot.png
- outputs/opportunity_scanner/opportunity_scan_manifest.json


In [ ]:

SITE_ID = "NHV"
SITE_NAME = "Newhaven ERF"
LAT = 50.80208
LON = 0.04961

DATE_FROM = "2024-01-01"
DATE_TO   = "2025-12-31"

PRODUCT_KEY = "NO2"
TIMELINESS = "OFFL"
BUFFER_DEG = 0.20
MAX_SCENES = 1000

OUTPUT_DIR = "outputs/opportunity_scanner"

GOOD_CLOUD_MAX = 20.0
GOOD_WIND_MAX = 3.0
TOP_N = 25


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import os, json, time, hashlib
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path(OUTPUT_DIR) / "weather_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars, retries=4, timeout=45):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": hourly_vars,
        "timezone": "UTC",
    }
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            print(f"Weather fetch failed (attempt {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Open-Meteo request failed after {retries} attempts: {last_err}")

def weather_cache_path(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    safe = f"{lat}_{lon}_{start_date}_{end_date}_{hourly_vars}".replace(",", "_").replace("/", "_").replace(":", "_")
    return cache_dir / f"{safe}.json"

def fetch_weather_cached(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    p = weather_cache_path(cache_dir, lat, lon, start_date, end_date, hourly_vars)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    j = fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars)
    p.write_text(json.dumps(j), encoding="utf-8")
    return j

print("UTC now:", datetime.now(timezone.utc).isoformat())


In [ ]:

CDSE_USERNAME = os.getenv("CDSE_USERNAME") or os.getenv("CDSE_USER")
CDSE_PASSWORD = os.getenv("CDSE_PASSWORD")

assert CDSE_USERNAME, "Missing CDSE_USERNAME/CDSE_USER secret"
assert CDSE_PASSWORD, "Missing CDSE_PASSWORD secret"

TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
ODATA_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"

def get_cdse_token(username, password):
    data = {
        "client_id": "cdse-public",
        "username": username,
        "password": password,
        "grant_type": "password",
    }
    r = requests.post(TOKEN_URL, data=data, timeout=60)
    r.raise_for_status()
    return r.json()["access_token"]

token = get_cdse_token(CDSE_USERNAME, CDSE_PASSWORD)


In [ ]:

PRODUCT_PATTERNS = {
    ("NO2", "OFFL"): "L2__NO2___",
    ("NO2", "NRTI"): "L2__NO2___",
    ("SO2", "OFFL"): "L2__SO2___",
    ("SO2", "NRTI"): "L2__SO2___",
    ("CO",  "OFFL"): "L2__CO____",
    ("CO",  "NRTI"): "L2__CO____",
}
name_pat = PRODUCT_PATTERNS[(PRODUCT_KEY.upper(), TIMELINESS.upper())]

minx, maxx = LON - BUFFER_DEG, LON + BUFFER_DEG
miny, maxy = LAT - BUFFER_DEG, LAT + BUFFER_DEG
wkt = f"POLYGON(({minx} {miny},{maxx} {miny},{maxx} {maxy},{minx} {maxy},{minx} {miny}))"

flt = (
    "Collection/Name eq 'SENTINEL-5P' "
    f"and ContentDate/Start ge {DATE_FROM}T00:00:00.000Z "
    f"and ContentDate/Start le {DATE_TO}T23:59:59.999Z "
    f"and OData.CSC.Intersects(area=geography'SRID=4326;{wkt}') "
    f"and contains(Name,'{name_pat}') "
    f"and contains(Name,'_{TIMELINESS.upper()}_')"
)

params = {"$filter": flt, "$top": str(MAX_SCENES), "$orderby": "ContentDate/Start asc", "$count": "true"}
headers = {"Authorization": f"Bearer {token}"}
r = requests.get(ODATA_URL, params=params, headers=headers, timeout=180)
r.raise_for_status()
value = r.json().get("value", [])

scene_rows = []
for p in value:
    scene_rows.append({
        "site_id": SITE_ID,
        "site_name": SITE_NAME,
        "lat": LAT,
        "lon": LON,
        "product_key": PRODUCT_KEY.upper(),
        "timeliness": TIMELINESS.upper(),
        "product_id": p.get("Id"),
        "name": p.get("Name"),
        "start_utc": p.get("ContentDate", {}).get("Start"),
        "end_utc": p.get("ContentDate", {}).get("End"),
        "published_utc": p.get("PublicationDate"),
        "online": p.get("Online"),
        "size_bytes": p.get("ContentLength"),
    })

scene_df = pd.DataFrame(scene_rows)
if not scene_df.empty:
    scene_df["start_utc"] = pd.to_datetime(scene_df["start_utc"], utc=True, errors="coerce")
    scene_df["date"] = scene_df["start_utc"].dt.date.astype("string")

scene_catalog_csv = Path(OUTPUT_DIR) / "opportunity_scene_catalog.csv"
scene_df.to_csv(scene_catalog_csv, index=False)

print("Scenes returned:", len(scene_df))
display(scene_df.head(20))


In [ ]:

j = fetch_weather_cached(CACHE_DIR, LAT, LON, DATE_FROM, DATE_TO, "wind_speed_10m,wind_direction_10m,cloud_cover,visibility")
h = j.get("hourly", {})

wx = pd.DataFrame({
    "time_utc": pd.to_datetime(h.get("time", []), utc=True, errors="coerce"),
    "wind_speed_ms": h.get("wind_speed_10m", []),
    "wind_dir_deg": h.get("wind_direction_10m", []),
    "cloud_cover_pct": h.get("cloud_cover", []),
    "visibility_m": h.get("visibility", []),
})

weather_csv = Path(OUTPUT_DIR) / "opportunity_weather_hourly.csv"
wx.to_csv(weather_csv, index=False)

display(wx.head(20))


In [ ]:

def nearest_hour(ts, weather_df):
    if weather_df.empty or pd.isna(ts):
        return pd.Series({
            "wind_speed_ms": np.nan,
            "wind_dir_deg": np.nan,
            "cloud_cover_pct": np.nan,
            "visibility_m": np.nan,
            "wx_time_utc": pd.NaT,
            "time_mismatch_hours": np.nan,
        })
    idx = (weather_df["time_utc"] - ts).abs().idxmin()
    row = weather_df.loc[idx]
    return pd.Series({
        "wind_speed_ms": row.get("wind_speed_ms"),
        "wind_dir_deg": row.get("wind_dir_deg"),
        "cloud_cover_pct": row.get("cloud_cover_pct"),
        "visibility_m": row.get("visibility_m"),
        "wx_time_utc": row.get("time_utc"),
        "time_mismatch_hours": abs((row.get("time_utc") - ts).total_seconds()) / 3600.0 if pd.notna(row.get("time_utc")) else np.nan,
    })

if not scene_df.empty:
    wx_match = scene_df["start_utc"].apply(lambda ts: nearest_hour(ts, wx))
    scene_eval = pd.concat([scene_df, wx_match], axis=1)
else:
    scene_eval = pd.DataFrame(columns=["date","product_id","start_utc","wind_speed_ms","wind_dir_deg","cloud_cover_pct","visibility_m","time_mismatch_hours"])

if not scene_eval.empty:
    scene_eval["scene_present"] = 1
    scene_eval["good_cloud_flag"] = (pd.to_numeric(scene_eval["cloud_cover_pct"], errors="coerce").fillna(100) <= GOOD_CLOUD_MAX).astype(int)
    scene_eval["good_wind_flag"] = (pd.to_numeric(scene_eval["wind_speed_ms"], errors="coerce").fillna(99) <= GOOD_WIND_MAX).astype(int)
    scene_eval["opportunity_score"] = (
        5.0
        + (5.0 - pd.to_numeric(scene_eval["wind_speed_ms"], errors="coerce").fillna(5)).clip(lower=0)
        + ((100.0 - pd.to_numeric(scene_eval["cloud_cover_pct"], errors="coerce").fillna(100)).clip(lower=0) / 20.0)
        + (pd.to_numeric(scene_eval["visibility_m"], errors="coerce").fillna(0) / 10000.0)
        + scene_eval["good_cloud_flag"] * 2.0
        + scene_eval["good_wind_flag"] * 2.0
    )

scene_eval_csv = Path(OUTPUT_DIR) / "opportunity_scene_evaluation.csv"
scene_eval.to_csv(scene_eval_csv, index=False)

display(scene_eval.head(20))


In [ ]:

if not scene_eval.empty:
    daily = (
        scene_eval.groupby("date", dropna=False)
        .agg(
            scenes=("product_id", "count"),
            first_overpass_utc=("start_utc", "min"),
            mean_wind_speed_ms=("wind_speed_ms", "mean"),
            mean_cloud_cover_pct=("cloud_cover_pct", "mean"),
            max_visibility_m=("visibility_m", "max"),
            best_opportunity_score=("opportunity_score", "max"),
            mean_opportunity_score=("opportunity_score", "mean"),
            good_cloud_any=("good_cloud_flag", "max"),
            good_wind_any=("good_wind_flag", "max"),
        )
        .reset_index()
    )
else:
    daily = pd.DataFrame(columns=[
        "date","scenes","first_overpass_utc","mean_wind_speed_ms","mean_cloud_cover_pct",
        "max_visibility_m","best_opportunity_score","mean_opportunity_score","good_cloud_any","good_wind_any"
    ])

daily["priority_flag"] = (
    (pd.to_numeric(daily["scenes"], errors="coerce").fillna(0) > 0)
    & (pd.to_numeric(daily["mean_cloud_cover_pct"], errors="coerce").fillna(100) <= GOOD_CLOUD_MAX)
    & (pd.to_numeric(daily["mean_wind_speed_ms"], errors="coerce").fillna(99) <= GOOD_WIND_MAX)
).astype(int)

ranked = daily.sort_values(["priority_flag", "best_opportunity_score", "scenes"], ascending=[False, False, False]).reset_index(drop=True)

daily_csv = Path(OUTPUT_DIR) / "opportunity_scan_daily.csv"
ranked_csv = Path(OUTPUT_DIR) / "opportunity_scan_ranked.csv"
daily.to_csv(daily_csv, index=False)
ranked.to_csv(ranked_csv, index=False)

display(ranked.head(TOP_N))


In [ ]:

fig, ax = plt.subplots(figsize=(11, 5))

if not ranked.empty:
    top_plot = ranked.head(min(TOP_N, len(ranked))).copy()
    x = pd.to_datetime(top_plot["date"])
    ax.plot(x, top_plot["best_opportunity_score"], marker="o", label="best_opportunity_score")
    ax.plot(x, top_plot["scenes"], marker="o", label="scene_count")

ax.set_title("Satellite opportunity scanner")
ax.set_xlabel("Date")
ax.set_ylabel("Score / scene count")
if not ranked.empty:
    ax.legend()
fig.autofmt_xdate()
fig.tight_layout()

plot_path = Path(OUTPUT_DIR) / "opportunity_scan_plot.png"
fig.savefig(plot_path, dpi=150)
plt.show()

print("Saved:", plot_path)


In [ ]:

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "site_id": SITE_ID,
    "site_name": SITE_NAME,
    "lat": LAT,
    "lon": LON,
    "date_from": DATE_FROM,
    "date_to": DATE_TO,
    "product_key": PRODUCT_KEY,
    "timeliness": TIMELINESS,
    "buffer_deg": BUFFER_DEG,
    "scene_count": int(len(scene_df)),
    "evaluated_scene_rows": int(len(scene_eval)),
    "daily_rows": int(len(daily)),
    "ranked_rows": int(len(ranked)),
    "outputs": {
        "scene_catalog_csv": str(scene_catalog_csv),
        "weather_csv": str(weather_csv),
        "scene_eval_csv": str(scene_eval_csv),
        "daily_csv": str(daily_csv),
        "ranked_csv": str(ranked_csv),
        "plot_png": str(plot_path),
    },
}
manifest_path = Path(OUTPUT_DIR) / "opportunity_scan_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Saved:", manifest_path)
